In [2]:
# Célula 1: Instalação das bibliotecas e criação das pastas
!pip install scikit-fuzzy pytest matplotlib numpy

import os
os.makedirs("AULA_11/src", exist_ok=True)
!touch AULA_11/src/__init__.py
print("Estrutura de diretórias criada com sucesso!")

Estrutura de diretórias criada com sucesso!


In [3]:
%%writefile AULA_11/src/engine.py
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

def construir_motor_inferencia():
    # 1. Definição das Variáveis e Universos de Discurso
    impacto = ctrl.Antecedent(np.arange(0, 101, 1), 'impacto')
    explorabilidade = ctrl.Antecedent(np.arange(0, 11, 1), 'explorabilidade')
    urgencia = ctrl.Consequent(np.arange(0, 101, 1), 'urgencia')

    # 2. Funções de Pertinência
    impacto['Baixo'] = fuzz.trimf(impacto.universe, [0, 0, 40])
    impacto['Médio'] = fuzz.trimf(impacto.universe, [20, 50, 80])
    impacto['Alto']  = fuzz.trimf(impacto.universe, [60, 100, 100])

    explorabilidade['Baixa']    = fuzz.trimf(explorabilidade.universe, [0, 0, 4])
    explorabilidade['Moderada'] = fuzz.trimf(explorabilidade.universe, [2, 5, 8])
    explorabilidade['Alta']     = fuzz.trimf(explorabilidade.universe, [6, 10, 10])

    urgencia['Urgência Baixa']   = fuzz.trimf(urgencia.universe, [0, 0, 33])
    urgencia['Urgência Média']   = fuzz.trimf(urgencia.universe, [15, 33, 66])
    urgencia['Urgência Alta']    = fuzz.trimf(urgencia.universe, [50, 66, 85])
    urgencia['Urgência Crítica'] = fuzz.trimf(urgencia.universe, [66, 100, 100])

    # 3. Base de Regras (3x3)
    rule1 = ctrl.Rule(impacto['Baixo'] & explorabilidade['Baixa'],    urgencia['Urgência Baixa'])
    rule2 = ctrl.Rule(impacto['Baixo'] & explorabilidade['Moderada'], urgencia['Urgência Baixa'])
    rule3 = ctrl.Rule(impacto['Baixo'] & explorabilidade['Alta'],     urgencia['Urgência Média'])

    rule4 = ctrl.Rule(impacto['Médio'] & explorabilidade['Baixa'],    urgencia['Urgência Média'])
    rule5 = ctrl.Rule(impacto['Médio'] & explorabilidade['Moderada'], urgencia['Urgência Alta'])
    rule6 = ctrl.Rule(impacto['Médio'] & explorabilidade['Alta'],     urgencia['Urgência Alta'])

    rule7 = ctrl.Rule(impacto['Alto']  & explorabilidade['Baixa'],    urgencia['Urgência Crítica'])
    rule8 = ctrl.Rule(impacto['Alto']  & explorabilidade['Moderada'], urgencia['Urgência Crítica'])
    rule9 = ctrl.Rule(impacto['Alto']  & explorabilidade['Alta'],     urgencia['Urgência Crítica'])

    # 4. Sistema de Controle
    urgencia_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9])
    return ctrl.ControlSystemSimulation(urgencia_ctrl)

Writing AULA_11/src/engine.py


In [4]:
%%writefile AULA_11/src/uti_triagem.py
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

def criar_sistema_triagem_uti():
    # 1. Variáveis
    freq_cardiaca = ctrl.Antecedent(np.arange(40, 161, 1), 'freq_cardiaca')
    spo2 = ctrl.Antecedent(np.arange(70, 101, 1), 'spo2')
    risco = ctrl.Consequent(np.arange(0, 101, 1), 'risco')

    risco.defuzzify_method = 'centroid'

    # 2. Funções de Pertinência
    freq_cardiaca['bradicardia'] = fuzz.trimf(freq_cardiaca.universe, [40, 40, 60])
    freq_cardiaca['normal']      = fuzz.trimf(freq_cardiaca.universe, [50, 75, 100])
    freq_cardiaca['taquicardia']  = fuzz.trapmf(freq_cardiaca.universe, [90, 110, 160, 160])

    spo2['critica']   = fuzz.trapmf(spo2.universe, [70, 70, 88, 91])
    spo2['limitrofe'] = fuzz.trimf(spo2.universe, [89, 93, 96])
    spo2['normal']    = fuzz.trapmf(spo2.universe, [94, 97, 100, 100])

    risco['baixo']    = fuzz.trimf(risco.universe, [0, 0, 35])
    risco['moderado'] = fuzz.trimf(risco.universe, [25, 50, 75])
    risco['critico']  = fuzz.trapmf(risco.universe, [65, 85, 100, 100])

    # 3. Regras Clínicas
    r1 = ctrl.Rule(spo2['critica'], risco['critico'])
    r2 = ctrl.Rule(freq_cardiaca['bradicardia'] & spo2['limitrofe'], risco['critico'])
    r3 = ctrl.Rule(freq_cardiaca['taquicardia'] & spo2['limitrofe'], risco['critico'])
    r4 = ctrl.Rule(freq_cardiaca['normal'] & spo2['limitrofe'], risco['moderado'])
    r5 = ctrl.Rule(freq_cardiaca['normal'] & spo2['normal'], risco['baixo'])

    uti_ctrl = ctrl.ControlSystem([r1, r2, r3, r4, r5])
    return ctrl.ControlSystemSimulation(uti_ctrl)

def avaliar_paciente(simulador, bpm, spo2_val):
    if not (40 <= bpm <= 160):
        raise ValueError(f"Frequência cardíaca fora do intervalo fisiológico [40, 160]: {bpm}")
    if not (70 <= spo2_val <= 100):
        raise ValueError(f"Saturação de Oxigénio SpO2 fora do intervalo [70, 100]: {spo2_val}")

    simulador.input['freq_cardiaca'] = bpm
    simulador.input['spo2'] = spo2_val
    simulador.compute()
    return simulador.output['risco']

if __name__ == "__main__":
    sim = criar_sistema_triagem_uti()
    print("Sistema de UTI instanciado com sucesso!")

Writing AULA_11/src/uti_triagem.py


In [13]:
%%writefile AULA_11/test_fuzzy_engine.py
import time
import pytest
import numpy as np
from src.engine import construir_motor_inferencia

@pytest.fixture
def simulador():
    return construir_motor_inferencia()

def executar_inferencia(simulador, impacto, explorabilidade):
    if not (0 <= impacto <= 100):
        raise ValueError(f"Impacto fora da escala [0, 100]: {impacto}")
    if not (0 <= explorabilidade <= 10):
        raise ValueError(f"Explorabilidade fora da escala [0, 10]: {explorabilidade}")

    simulador.input['impacto'] = impacto
    simulador.input['explorabilidade'] = explorabilidade
    simulador.compute()
    return simulador.output['urgencia']

def test_saida_dentro_do_universo(simulador):
    casos = [(0, 0), (50, 5), (100, 10), (30, 8), (80, 2)]
    for imp, exp in casos:
        res = executar_inferencia(simulador, imp, exp)
        assert 0.0 <= res <= 100.0, f"Falha: Saída {res} fora de [0, 100]"

def test_monotonicidade_impacto(simulador):
    exp_fixa = 5.0
    urg_baixa = executar_inferencia(simulador, 20.0, exp_fixa)
    urg_media = executar_inferencia(simulador, 50.0, exp_fixa)
    urg_alta = executar_inferencia(simulador, 90.0, exp_fixa)
    assert urg_baixa <= urg_media <= urg_alta

def test_entradas_invalidas_disparam_erro(simulador):
    with pytest.raises(ValueError):
        executar_inferencia(simulador, -10.0, 5.0)
    with pytest.raises(ValueError):
        executar_inferencia(simulador, 150.0, 5.0)
    with pytest.raises(ValueError):
        executar_inferencia(simulador, 50.0, 12.0)

def test_latencia_inferencia(simulador):
    inicio = time.time()
    for _ in range(100):
        imp = np.random.uniform(0, 100)
        exp = np.random.uniform(0, 10)
        executar_inferencia(simulador, imp, exp)
    tempo_total = time.time() - inicio
    tempo_medio_ms = (tempo_total / 100) * 1000
    print(f"\nTempo médio por inferência: {tempo_medio_ms:.3f} ms")
    assert tempo_medio_ms < 10.0

Overwriting AULA_11/test_fuzzy_engine.py


In [16]:
%%writefile AULA_11/test_uti_triagem.py
import pytest
from src.uti_triagem import criar_sistema_triagem_uti, avaliar_paciente

@pytest.fixture
def simulador_uti():
    return criar_sistema_triagem_uti()

def test_paciente_saudavel_baixo_risco(simulador_uti):
    risco = avaliar_paciente(simulador_uti, 75, 98)
    assert risco < 30.0, f"Risco esperado < 30.0%, obtido: {risco:.2f}%"

def test_salto_nao_linear_queda_spo2(simulador_uti):
    risco_normal = avaliar_paciente(simulador_uti, 75, 95)
    risco_queda = avaliar_paciente(simulador_uti, 75, 89)

    assert risco_normal < 38.0, f"Risco SpO2 95% esperado < 38.0%, obtido: {risco_normal:.2f}%"
    assert risco_queda > 65.0, f"Risco SpO2 89% esperado > 65.0%, obtido: {risco_queda:.2f}%"
    assert risco_queda > risco_normal, "O risco deve aumentar com a queda do SpO2"

def test_validacao_limites_fisiologicos(simulador_uti):
    with pytest.raises(ValueError):
        avaliar_paciente(simulador_uti, 35, 95)

    with pytest.raises(ValueError):
        avaliar_paciente(simulador_uti, 75, 105)

Overwriting AULA_11/test_uti_triagem.py


In [7]:
# Célula para Rodada 1 - COMPLETO
!export PYTHONPATH=$PYTHONPATH:$(pwd)/AULA_11 && pytest -v AULA_11/test_fuzzy_engine.py
!export PYTHONPATH=$PYTHONPATH:$(pwd)/AULA_11 && pytest -s AULA_11/test_fuzzy_engine.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12
collected 4 items                                                              

AULA_11/test_fuzzy_engine.py::test_saida_dentro_do_universo PASSED       [ 25%]
AULA_11/test_fuzzy_engine.py::test_monotonicidade_impacto PASSED         [ 50%]
AULA_11/test_fuzzy_engine.py::test_entradas_invalidas_disparam_erro PASSED [ 75%]
AULA_11/test_fuzzy_engine.py::test_latencia_inferencia PASSED            [100%]

============================== 4 passed in 2.48s ===============================
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0

In [12]:
# Execução da Rodada 2 (deve ter erro)- COMPLETO
!export PYTHONPATH=$PYTHONPATH:$(pwd)/AULA_11 && pytest -v AULA_11/test_fuzzy_engine.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12
collected 4 items                                                              

AULA_11/test_fuzzy_engine.py::test_saida_dentro_do_universo PASSED       [ 25%]
AULA_11/test_fuzzy_engine.py::test_monotonicidade_impacto PASSED         [ 50%]
AULA_11/test_fuzzy_engine.py::test_entradas_invalidas_disparam_erro PASSED [ 75%]
AULA_11/test_fuzzy_engine.py::test_latencia_inferencia FAILED            [100%]

=================================== FAILURES ===================================
___________________________ test_latencia_inferencia ___________________________

simulador = <skfuzzy.control.controlsystem.ControlSystemSimulation object at 0x7a3883636060>

    def test_latencia_inferencia(simulador):
        ini

In [14]:
# Execução da Rodada 3 (Erro corrigido) - COMPLETO
!export PYTHONPATH=$PYTHONPATH:$(pwd)/AULA_11 && pytest -v AULA_11/test_fuzzy_engine.py
!export PYTHONPATH=$PYTHONPATH:$(pwd)/AULA_11 && pytest -s AULA_11/test_fuzzy_engine.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12
collected 4 items                                                              

AULA_11/test_fuzzy_engine.py::test_saida_dentro_do_universo PASSED       [ 25%]
AULA_11/test_fuzzy_engine.py::test_monotonicidade_impacto PASSED         [ 50%]
AULA_11/test_fuzzy_engine.py::test_entradas_invalidas_disparam_erro PASSED [ 75%]
AULA_11/test_fuzzy_engine.py::test_latencia_inferencia PASSED            [100%]

============================== 4 passed in 2.57s ===============================
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0

In [17]:
# Célula para rodar os testes da UTI
!export PYTHONPATH=$PYTHONPATH:$(pwd)/AULA_11 && pytest -v AULA_11/test_uti_triagem.py
!export PYTHONPATH=$PYTHONPATH:$(pwd)/AULA_11 && pytest -s AULA_11/test_uti_triagem.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12
collected 3 items                                                              

AULA_11/test_uti_triagem.py::test_paciente_saudavel_baixo_risco PASSED   [ 33%]
AULA_11/test_uti_triagem.py::test_salto_nao_linear_queda_spo2 PASSED     [ 66%]
AULA_11/test_uti_triagem.py::test_validacao_limites_fisiologicos PASSED  [100%]

============================== 3 passed in 1.38s ===============================
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12
collected 3 items                                                          

In [18]:
# Compactar e descarregar a pasta do projeto
from google.colab import files
import shutil

# Criar ficheiro ZIP da pasta
shutil.make_archive('AULA_11', 'zip', 'AULA_11')

# Descarregar automaticamente o ZIP
files.download('AULA_11.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>